# 长期记忆-agent

## 1、在工具中访问长期记忆

### 1.1 基于InMemoryStore

In [ ]:
import os

from fastmcp.utilities.docstring_parsing import parse_docstring
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware, TodoListMiddleware, wrap_model_call, \
    ModelRequest, ModelResponse, wrap_tool_call
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.stores import InMemoryStore
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.prebuilt.tool_node import ToolCallRequest
from openai.types.beta.realtime import response_text_done_event
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # profile={
    #     "max_input_tokens": 1_000_000
    # },
    # 关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)


In [ ]:
from langchain_core.tools import tool
from typing import NotRequired
from langgraph.prebuilt import ToolRuntime
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState


store = InMemoryStore()
#自定义一个继承于AgentState的类
class CustomState(AgentState):
    user_id:NotRequired[str]
#保存用户信息到长期记忆
@tool(parse_docstring=True)
def save_user_info(name:str,runtime:ToolRuntime)->str:
    """
    将客户信息保存在长期记忆中

    Args:
        name:用户名
        runtime:工具的运行时

    Returns:
        str:保存状态
    """
    namespace=("users",)
    key=runtime.state["user_id"]
    value={"name":name}
    runtime.store.put(namespace,key,value)
    return "saved"
@tool(parse_docstring=True)
def get_user_info(runtime:ToolRuntime)->str:
    """
    从长期记忆中读取客户的信息

    Args:
        runtime:工具的运行时

    Returns:
        str:用户信息
    """
    namespace=("users",)
    key=runtime.state["user_id"]
    item=runtime.store.get(namespace,key)
    return str(item.value) if item else "unknown"
agent=create_agent(
        model=model,
        tools=[save_user_info, get_user_info],
        store=store,
        state_schema=CustomState,
        system_prompt="用户提及个人信息时，可以使用工具保存用户信息。如果用户询问个人信息时，可以尝试使用工具读取用户信息"
    )
print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()